In [ ]:
PROVIDER="gpt" # claude # llama # gemini

In [ ]:
import sys
import importlib

import pandas as pd
from os.path import join
sys.path.append("..")

from config import DATA_DIR, PROJECT_DIR
from utils.data_helpers import convert_to_int
from utils.classification_helpers import import_lexicon_classification_files, map_lexicon_chapters_to_ihra_sections, map_decoding_codes, split_ambiguous_sections

classification_helpers_module = importlib.import_module("utils.classification_helpers")

extract_ihra_ref = getattr(classification_helpers_module, f"extract_ihra_ref_{PROVIDER}")
extract_tax_ref = getattr(classification_helpers_module, f"extract_tax_ref_{PROVIDER}")




# upper bound of total number of records per data source, in 50-steps
total_b = 1900
total_d = 3000

In [3]:
df_d_1_raw = pd.read_feather(join(DATA_DIR, "decoding_label_1_text.feather"))
df_d_0_raw = pd.read_feather(join(DATA_DIR, "decoding_label_0_text.feather"))
df_b_1_raw = pd.read_feather(join(DATA_DIR, "bloomington_label_1_text.feather"))
df_b_0_raw = pd.read_feather(join(DATA_DIR, "bloomington_label_0_text.feather"))

In [4]:
df_d_1_raw.sample(1)

,comment_cleaned,id,discourse,comment_level,comment_codes_all,source_outlet
1362,Israel needs to be dissolved,1363,21_AICN,1.0,"I1, L13d2",INDEP


## Taxonomy

In [153]:
tax_d_1 = import_lexicon_classification_files("decoding", label=1, total=total_d, verbose=False, kb="tax", provider=PROVIDER)
tax_d_0 = import_lexicon_classification_files("decoding", label=0, total=1000, verbose=False, kb="tax", provider=PROVIDER)
tax_b_1 = import_lexicon_classification_files("bloomington", label=1, total=total_b, verbose=False, kb="tax", provider=PROVIDER) # 
tax_b_0 = import_lexicon_classification_files("bloomington", label=0, total=1000, verbose=False, kb="tax", provider=PROVIDER)

In [154]:
tax_d_1.sample(2)

,classification,explanation
index,,
2040,No,This criticizes Israel as 'an apartheid state'...
573,No,The text is a generic insult directed at an un...


In [155]:
for df in [tax_b_0, tax_b_1, tax_d_0, tax_d_1]:
    print(len(df))
    print([i for i in range(len(df.index)) if i not in df.index])
    df.rename(
    columns={"classification": "classification_tax",
            "explanation": "explanation_tax"}, inplace=True)

1000
[]
1858
[]
992
[]
2977
[]


## Taxonomy + Examples

In [156]:
tax_ex_d_1 = import_lexicon_classification_files("decoding", label=1, total=total_d, verbose=False, kb="tax_ex", provider=PROVIDER)
tax_ex_d_0 = import_lexicon_classification_files("decoding", label=0, total=1000, verbose=False, kb="tax_ex", provider=PROVIDER)
tax_ex_b_1 = import_lexicon_classification_files("bloomington", label=1, total=total_b, verbose=False, kb="tax_ex", provider=PROVIDER) # 
tax_ex_b_0 = import_lexicon_classification_files("bloomington", label=0, total=1000, verbose=False, kb="tax_ex", provider=PROVIDER)

In [157]:
tax_b_1.sample(2)

,classification_tax,explanation_tax
index,,
1582,Yes,The post uses the slur 'ZioNazis' as a hostile...
1007,Yes,The text is antisemitic. It frames 'Jews' coll...


In [158]:
for df in [tax_ex_b_0, tax_ex_b_1, tax_ex_d_0, tax_ex_d_1]:
    print(len(df))
    print([i for i in range(len(df.index)) if i not in df.index])
    df.rename(
    columns={"classification": "classification_tax_ex",
            "explanation": "explanation_tax_ex"}, inplace=True)

1000
[]
1858
[]
992
[]
2977
[]


## IHRA

In [159]:
ihra_d_1 = import_lexicon_classification_files("decoding", label=1, total=total_d, verbose=False, kb="ihra", provider=PROVIDER)
ihra_d_0 = import_lexicon_classification_files("decoding", label=0, total=1000, verbose=False, kb="ihra", provider=PROVIDER)    
ihra_b_1 = import_lexicon_classification_files("bloomington", label=1, total=total_b, verbose=False, kb="ihra", provider=PROVIDER) #
ihra_b_0 = import_lexicon_classification_files("bloomington", label=0, total=1000, verbose=False, kb="ihra", provider=PROVIDER)    

In [160]:
ihra_d_1.sample(2)

,classification,explanation
index,,
1669,No,The text does not express hatred toward Jews o...
1472,No,The post criticizes Israelis/Israel for allege...


In [161]:
for df in [ihra_d_1, ihra_d_0, ihra_b_1, ihra_b_0]:
    print(len(df))
    print([i for i in range(len(df.index)) if i not in df.index])
    df.rename(
    columns={"classification": "classification_ihra_explanation_cleaned",
            "explanation": "explanation_ihra_explanation_cleaned"}, inplace=True)

2977
[]
992
[]
1858
[]
1000
[]


## No KB

In [162]:
no_kb_d_1 = import_lexicon_classification_files("decoding", label=1, total=total_d, verbose=False, kb="no_kb", provider=PROVIDER, flat_file=True, sort_index=False)
no_kb_d_0 = import_lexicon_classification_files("decoding", label=0, total=1000, verbose=False, kb="no_kb", provider=PROVIDER, flat_file=True, sort_index=False)    
no_kb_b_1 = import_lexicon_classification_files("bloomington", label=1, total=total_b, verbose=False, kb="no_kb", provider=PROVIDER, flat_file=True, sort_index=False) #
no_kb_b_0 = import_lexicon_classification_files("bloomington", label=0, total=1000, verbose=False, kb="no_kb", provider=PROVIDER, flat_file=True, sort_index=False)    

In [163]:
no_kb_b_0.index

Index([ 1,  2,  3,  4,  5,  6,  7,  8,  9, 10,
       ...
       41, 42, 43, 44, 45, 46, 47, 48, 49, 50],
      dtype='int64', name='index', length=1000)

In [164]:
for df in [no_kb_d_1, no_kb_d_0, no_kb_b_1, no_kb_b_0]:
    print(len(df))
    # print([i for i in range(len(df.index)) if i not in df.index])
    df.rename(
    columns={"classification": "classification_no_kb_cleaned"}, inplace=True)

2977
992
1858
1000


In [165]:
no_kb_d_1.reset_index(drop=True, inplace=True)
no_kb_d_0.reset_index(drop=True, inplace=True)
no_kb_b_1.reset_index(drop=True, inplace=True)
no_kb_b_0.reset_index(drop=True, inplace=True)

## Merge

In [166]:
df_d_1_tmp = pd.merge(df_d_1_raw, tax_d_1, how="left", left_index=True, right_index=True)
df_d_0_tmp = pd.merge(df_d_0_raw, tax_d_0, how="left", left_index=True, right_index=True)
df_b_1_tmp = pd.merge(df_b_1_raw, tax_b_1, how="left", left_index=True, right_index=True)
df_b_0_tmp = pd.merge(df_b_0_raw, tax_b_0, how="left", left_index=True, right_index=True)

In [167]:
df_d_1_tmp = pd.merge(df_d_1_tmp, tax_ex_d_1, how="left", left_index=True, right_index=True)
df_d_0_tmp = pd.merge(df_d_0_tmp, tax_ex_d_0, how="left", left_index=True, right_index=True)
df_b_1_tmp = pd.merge(df_b_1_tmp, tax_ex_b_1, how="left", left_index=True, right_index=True)
df_b_0_tmp = pd.merge(df_b_0_tmp, tax_ex_b_0, how="left", left_index=True, right_index=True)

In [168]:
df_d_1_tmp = pd.merge(df_d_1_tmp, ihra_d_1, how="left", left_index=True, right_index=True)
df_d_0_tmp = pd.merge(df_d_0_tmp, ihra_d_0, how="left", left_index=True, right_index=True)
df_b_1_tmp = pd.merge(df_b_1_tmp, ihra_b_1, how="left", left_index=True, right_index=True)
df_b_0_tmp = pd.merge(df_b_0_tmp, ihra_b_0, how="left", left_index=True, right_index=True)

In [169]:
df_d_1_tmp = pd.merge(df_d_1_tmp, no_kb_d_1, how="left", left_index=True, right_index=True)
df_d_0_tmp = pd.merge(df_d_0_tmp, no_kb_d_0, how="left", left_index=True, right_index=True)
df_b_1_tmp = pd.merge(df_b_1_tmp, no_kb_b_1, how="left", left_index=True, right_index=True)     
df_b_0_tmp = pd.merge(df_b_0_tmp, no_kb_b_0, how="left", left_index=True, right_index=True)

In [170]:
df_d_1_tmp.sample(2)

,comment_cleaned,id,discourse,comment_level,comment_codes_all,source_outlet,classification_tax,explanation_tax,classification_tax_ex,explanation_tax_ex,classification_ihra_explanation_cleaned,explanation_ihra_explanation_cleaned,classification_no_kb_cleaned
194,"Martyr Sheikh Ahmad Yassin, the founder of the...",195,23_AIC,0.0,"I1, S7b",GUARD,Yes,The post glorifies Hamas founder Ahmad Yassin ...,No,The post praises Hamas founder Ahmad Yassin an...,No,The post is highly hostile toward Israel and p...,No
2671,"Hitler loved Jews, that’s why he murdered us. ...",2672,22_UKR,0.0,"I1, M3, OAJ, D2c, Example for illustration, D2d",DAILY,No,This appears to be ironic or satirical comment...,No,This appears to be sarcastic or ironic comment...,No,This appears to be sarcastic or darkly ironic ...,No


In [171]:
for df in [df_d_1_tmp, df_d_0_tmp, df_b_1_tmp, df_b_0_tmp]:
    print(len(df))
    print([i for i in range(len(df.index)) if i not in df.index])

2977
[]
992
[]
1858
[]
1000
[]


In [172]:
# store as feather files
df_b_1_tmp.to_feather(join(DATA_DIR, f"{PROVIDER}_bloomington_label_1.feather"))
df_b_0_tmp.to_feather(join(DATA_DIR, f"{PROVIDER}_bloomington_label_0.feather"))
df_d_1_tmp.to_feather(join(DATA_DIR, f"{PROVIDER}_decoding_label_1.feather"))
df_d_0_tmp.to_feather(join(DATA_DIR, f"{PROVIDER}_decoding_label_0.feather"))


## CHAPTER AND SECTION EXTRACTION

In [173]:
df_d_1 = pd.read_feather(join(DATA_DIR, f"{PROVIDER}_decoding_label_1.feather"))
df_d_0 = pd.read_feather(join(DATA_DIR, f"{PROVIDER}_decoding_label_0.feather"))
df_b_1 = pd.read_feather(join(DATA_DIR, f"{PROVIDER}_bloomington_label_1.feather"))
df_b_0 = pd.read_feather(join(DATA_DIR, f"{PROVIDER}_bloomington_label_0.feather"))

In [174]:
for df in [df_d_1, df_d_0, df_b_1, df_b_0]:
    print(df.columns)

Index(['comment_cleaned', 'id', 'discourse', 'comment_level',
       'comment_codes_all', 'source_outlet', 'classification_tax',
       'explanation_tax', 'classification_tax_ex', 'explanation_tax_ex',
       'classification_ihra_explanation_cleaned',
       'explanation_ihra_explanation_cleaned', 'classification_no_kb_cleaned'],
      dtype='object')
Index(['comment_cleaned', 'id', 'discourse', 'comment_level',
       'comment_codes_all', 'source_outlet', 'classification_tax',
       'explanation_tax', 'classification_tax_ex', 'explanation_tax_ex',
       'classification_ihra_explanation_cleaned',
       'explanation_ihra_explanation_cleaned', 'classification_no_kb_cleaned'],
      dtype='object')
Index(['comment_cleaned', 'id', 'keyword', 'ihra_section_1', 'ihra_section_2',
       'classification_tax', 'explanation_tax', 'classification_tax_ex',
       'explanation_tax_ex', 'classification_ihra_explanation_cleaned',
       'explanation_ihra_explanation_cleaned', 'classification_no_kb

In [175]:
df_d_1.sample(2)

,comment_cleaned,id,discourse,comment_level,comment_codes_all,source_outlet,classification_tax,explanation_tax,classification_tax_ex,explanation_tax_ex,classification_ihra_explanation_cleaned,explanation_ihra_explanation_cleaned,classification_no_kb_cleaned
295,She is definitely right in respect of the fact...,296,23_DA,0.0,"I1, M3, OAJ, S7, D7, D8, SE7",INDEP,Yes,The text advances an antisemitic trope by clai...,Yes,The text invokes an antisemitic trope about di...,Yes,The post contains an antisemitic trope by clai...,Yes
2916,America and American organizations have been c...,2917,23_AIC,0.0,"I1, S2",BBC,No,The text is highly hostile toward Israel and t...,No,The text is highly hostile toward Israel and t...,No,The post harshly condemns Israel and the Unite...,No


In [ ]:
for df in [df_d_1, df_d_0, df_b_1, df_b_0]:
    df["explanation_ihra_explanation_sections"] = df.apply(lambda row: extract_ihra_ref(row["explanation_ihra_explanation_cleaned"]), axis=1)

In [ ]:
for df in [df_d_1, df_d_0, df_b_1, df_b_0]:
    df["explanation_tax_chapters"] = df.apply(lambda row: extract_tax_ref(row["explanation_tax"]), axis=1)

In [ ]:
for df in [df_d_1, df_d_0, df_b_1, df_b_0]:
    df["explanation_tax_ex_chapters"] = df.apply(lambda row: extract_tax_ref(row["explanation_tax_ex"]), axis=1)

In [ ]:
print(df_b_1.explanation_tax_ex_chapters.value_counts())

In [180]:
df_b_1.to_feather(join(DATA_DIR, f"{PROVIDER}_bloomington_label_1.feather"))
df_b_0.to_feather(join(DATA_DIR, f"{PROVIDER}_bloomington_label_0.feather"))
df_d_1.to_feather(join(DATA_DIR, f"{PROVIDER}_decoding_label_1.feather"))
df_d_0.to_feather(join(DATA_DIR, f"{PROVIDER}_decoding_label_0.feather"))

### Map decoding codes to lexicon chapters and IHRA sections


In [181]:
df_d_1 = pd.read_feather(join(DATA_DIR, f"{PROVIDER}_decoding_label_1.feather"))

In [182]:
decoding_lexicon_ihra = pd.read_csv(join(PROJECT_DIR, "llm_prompting", "external_ressources", "decoding_ihra_lexicon_mapping.csv"))
decoding_lexicon_ihra.head(2)

,decoding_code_detailled,chapter_lexicon,ihra_section,decoding_short
0,S1,2,2,a12
1,S2,3,27,a12


In [ ]:

decoding_lexicon_ihra.chapter_lexicon = decoding_lexicon_ihra.chapter_lexicon.map(convert_to_int)
decoding_lexicon_ihra.ihra_section = decoding_lexicon_ihra.ihra_section.map(convert_to_int)
# Entries in the column `decoding_code_detailled` are unique
decoding_lexicon = dict(
    zip(decoding_lexicon_ihra["decoding_code_detailled"], decoding_lexicon_ihra["chapter_lexicon"]))
# decoding_lexicon
df_d_1["comment_codes_all_list"] = df_d_1["comment_codes_all"].map(lambda x: [item.strip() for item in x.split(",")])
# map to lexicon chapters
df_d_1["comment_codes_all_chapters"] = df_d_1["comment_codes_all_list"].map(
    lambda x: map_decoding_codes(x, decoding_lexicon))
decoding_ihra = dict(
    zip(decoding_lexicon_ihra["decoding_code_detailled"], decoding_lexicon_ihra["ihra_section"]))
# decoding_ihra
# map to IHRA sections
df_d_1["comment_codes_all_sections"] = df_d_1["comment_codes_all_list"].map(
    lambda x: map_decoding_codes(x, decoding_ihra))
# clean up ambiguous IHRA sections
df_d_1["comment_codes_all_sections"] = df_d_1["comment_codes_all_sections"].map(split_ambiguous_sections)

In [ ]:
df_d_1.sample(2)

In [185]:
df_d_1.to_feather(join(DATA_DIR, f"{PROVIDER}_decoding_label_1.feather"))

## PREPARE FOR RQ3

In [ ]:
df_d_1 = pd.read_feather(join(DATA_DIR, f"{PROVIDER}_decoding_label_1.feather"))
df_b_1 = pd.read_feather(join(DATA_DIR, f"{PROVIDER}_bloomington_label_1.feather"))

In [ ]:
## Merge both annotator labels for Bloomington 
df_b_1["ihra_sections"] = df_b_1.apply(
    lambda x: list(set([x["ihra_section_1"]] + [x["ihra_section_2"]])), axis=1)

In [ ]:
## Count tax chapters 
df_b_1["explanation_tax_chapters_no"] = df_b_1["explanation_tax_chapters"].map(
    lambda x: len(x) if type(x)==list else 0)
df_b_1["explanation_tax_ex_chapters_no"] = df_b_1["explanation_tax_ex_chapters"].map(
    lambda x: len(x) if type(x)==list else 0)
df_d_1["explanation_tax_chapters_no"] = df_d_1["explanation_tax_chapters"].map(
    lambda x: len(x) if type(x)==list else 0)
df_d_1["explanation_tax_ex_chapters_no"] = df_d_1["explanation_tax_ex_chapters"].map(
    lambda x: len(x) if type(x)==list else 0)

In [ ]:
## Map tax chapters to IHRA sections 
ihra_lexicon_df = pd.read_feather(join(PROJECT_DIR, "llm_prompting", "external_ressources", "ihra_lexicon.feather"))
ihra_lexicon = dict(zip(ihra_lexicon_df["ihra_section"],ihra_lexicon_df["lexicon_chapter"]))
df_d_1["explanation_tax_sections"] = df_d_1["explanation_tax_chapters"].map(
    lambda x: map_lexicon_chapters_to_ihra_sections(x, ihra_lexicon))
df_b_1["explanation_tax_sections"] = df_b_1["explanation_tax_chapters"].map(
    lambda x: map_lexicon_chapters_to_ihra_sections(x, ihra_lexicon))

df_d_1["explanation_tax_ex_sections"] = df_d_1["explanation_tax_ex_chapters"].map(
    lambda x: map_lexicon_chapters_to_ihra_sections(x, ihra_lexicon))
df_b_1["explanation_tax_ex_sections"] = df_b_1["explanation_tax_ex_chapters"].map(
    lambda x: map_lexicon_chapters_to_ihra_sections(x, ihra_lexicon))

In [ ]:
df_b_1.to_feather(join(DATA_DIR, f"{PROVIDER}_bloomington_label_1.feather"))
df_d_1.to_feather(join(DATA_DIR, f"{PROVIDER}_decoding_label_1.feather"))